In [ ]:
import os, subprocess, time, shutil, sys
from pathlib import Path
from IPython.display import display, HTML, clear_output

# Lightning AI usually stores project files under /teamspace/studios/this_studio.
# If that path is not present, this falls back to the current notebook folder.
LIGHTNING_WORKSPACE = Path("/teamspace/studios/this_studio")
BASE_PATH = LIGHTNING_WORKSPACE if LIGHTNING_WORKSPACE.exists() else Path.cwd()
COMFY_PATH = str(BASE_PATH / "ComfyUI")

def run(cmd, cwd=None, check=True):
    display(HTML(f"<pre style='color:#9ad;font-size:12px'>Running: {' '.join(cmd)}</pre>"))
    return subprocess.run(cmd, cwd=cwd, check=check)

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[1/5] Pinning NumPy for compiled package compatibility...</p>"))
run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2"])
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch>=2.8.0", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128"])

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[2/5] Installing Blackwell-compatible PyTorch + dependencies...</p>"))
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "torch>=2.8.0", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128"])
run([sys.executable, "-m", "pip", "install", "-q", "torchsde", "einops", "diffusers", "accelerate", "av", "spandrel", "albumentations", "onnx", "opencv-python-headless", "onnxruntime-gpu", "tqdm", "Pillow", "requests", "gradio"])

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[3/5] Setting up ComfyUI backend...</p>"))
if not os.path.exists(COMFY_PATH):
    run(["git", "clone", "-q", "https://github.com/comfyanonymous/ComfyUI", COMFY_PATH])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY_PATH}/requirements.txt"])

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[4/5] Installing custom nodes...</p>"))
nodes_dir = f"{COMFY_PATH}/custom_nodes"
os.makedirs(nodes_dir, exist_ok=True)
for node_url in [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/Lightricks/ComfyUI-LTXVideo/",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/kijai/ComfyUI-MelBandRoFormer"
]:
    name = node_url.rstrip("/").split("/")[-1]
    path = os.path.join(nodes_dir, name)
    if not os.path.exists(path):
        run(["git", "clone", "-q", node_url, path])
    req = f"{path}/requirements.txt"
    if os.path.exists(req):
        run([sys.executable, "-m", "pip", "install", "-q", "-r", req])

run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2"])

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[5/5] Downloading LTX-2.3 weights...</p>"))

if shutil.which("aria2c") is None:
    try:
        run(["apt-get", "update", "-qq"], check=False)
        run(["apt-get", "install", "-y", "-qq", "aria2"], check=False)
    except Exception:
        pass

def dl(url, dest, fname):
    import requests
    Path(dest).mkdir(parents=True, exist_ok=True)
    fpath = os.path.join(dest, fname)
    if os.path.exists(fpath):
        return
    if shutil.which("aria2c"):
        run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", "-d", dest, "-o", fname, url])
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        tmp = fpath + ".part"
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        os.replace(tmp, fpath)

B = COMFY_PATH + "/models"
dl("https://huggingface.co/vantagewithai/LTX2.3-10Eros-GGUF/resolve/main/10Eros_v1-Q4_K_M.gguf", f"{B}/unet", "10Eros_v1-Q4_K_M.gguf")
dl("https://huggingface.co/llmfan46/gemma-3-12b-it-ultra-uncensored-heretic-GGUF/resolve/main/gemma-3-12b-it-heretic-IQ4_XS.gguf", f"{B}/text_encoders", "gemma-3-12b-it-heretic-IQ4_XS.gguf")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors", f"{B}/text_encoders", "ltx-2.3-22b-dev_embeddings_connectors.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors", f"{B}/vae", "ltx-2.3-22b-dev_video_vae.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_audio_vae.safetensors", f"{B}/vae", "ltx-2.3-22b-dev_audio_vae.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors", f"{B}/latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
dl("https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors", f"{B}/diffusion_models", "MelBandRoformer_fp16.safetensors")
dl("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/taeltx2_3.safetensors", f"{B}/vae", "taeltx2_3.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384-1.1.safetensors", f"{B}/loras", "ltx-2.3-22b-distilled-lora-384-1.1.safetensors")

clear_output()
display(HTML(f"<div style='padding:15px;background:#e8f5e9;border-left:5px solid #4caf50;border-radius:4px;color:#2e7d32;font-family:sans-serif;'><b>✨ Initialization Complete!</b><br>ComfyUI path: <code>{COMFY_PATH}</code><br>Using distilled LoRA: <code>ltx-2.3-22b-distilled-lora-384-1.1.safetensors</code></div>"))


In [ ]:
import gradio as gr
import json, urllib.request, urllib.error, time, os, glob, socket, shutil, subprocess, threading
from pathlib import Path
from PIL import Image

LIGHTNING_WORKSPACE = Path("/teamspace/studios/this_studio")
BASE_PATH = LIGHTNING_WORKSPACE if LIGHTNING_WORKSPACE.exists() else Path.cwd()
COMFY_PATH  = str(BASE_PATH / "ComfyUI")
COMFY_LOG_PATH = str(BASE_PATH / "comfyui_server.log")
OUTPUT_PATH = f"{COMFY_PATH}/output"
INPUT_PATH  = f"{COMFY_PATH}/input"
LORA_11_NAME = "ltx-2.3-22b-distilled-lora-384-1.1.safetensors"

def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

def boot_server():
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    log = open(COMFY_LOG_PATH, "a", encoding="utf-8")
    proc = subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd=COMFY_PATH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def stream_logs():
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

    threading.Thread(target=stream_logs, daemon=True).start()
    start_time = time.time()
    while not is_server_running():
        if time.time() - start_time > 120:
            raise RuntimeError(f"Backend server failed to start within 2 minutes. Check {COMFY_LOG_PATH}.")
        time.sleep(2)

def load_workflow():
    url = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

def queue_prompt(wf):
    data = json.dumps({"prompt": wf}).encode("utf-8")
    req  = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    try:
        return json.loads(urllib.request.urlopen(req).read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"API ERROR: {e.read().decode()}")

def get_latest_video():
    mp4s = glob.glob(f"{OUTPUT_PATH}/**/*.mp4", recursive=True) + glob.glob(f"{OUTPUT_PATH}/*.mp4")
    if not mp4s:
        return None
    return max(mp4s, key=os.path.getctime)

def generate_video(mode, image_filepath, prompt, width, height, duration, seed, progress=gr.Progress()):
    progress(0, desc="Starting Server...")
    if not os.path.exists(COMFY_PATH):
        raise gr.Error("Engine not found. Run Step 1 first, then launch this UI on a strong GPU.")
    if not is_server_running():
        boot_server()
    os.makedirs(INPUT_PATH, exist_ok=True)

    W = max(256, round(width / 32) * 32)
    H = max(256, round(height / 32) * 32)
    wf = load_workflow()
    wf["345"]["inputs"]["unet_name"] = "10Eros_v1-Q4_K_M.gguf"
    wf["346"]["inputs"]["clip_name1"] = "gemma-3-12b-it-heretic-IQ4_XS.gguf"

    wf["292"]["inputs"]["value"] = W
    wf["293"]["inputs"]["value"] = H
    wf["285"]["inputs"]["value"] = 24
    wf["121"]["inputs"]["text"] = prompt
    wf["291"]["inputs"]["value"] = duration
    wf["134"]["inputs"]["lora_name"] = LORA_11_NAME
    wf["137"]["inputs"]["sampler_name"] = "lcm"
    wf["360"]["inputs"]["sigmas"] = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    wf["129"]["inputs"]["cfg"] = 1.0
    wf["115"]["inputs"]["noise_seed"] = int(seed)
    wf["138"]["inputs"]["sampler_name"] = "euler_cfg_pp"
    wf["359"]["inputs"]["sigmas"] = "0.85, 0.7250, 0.4219, 0.0"
    wf["103"]["inputs"]["cfg"] = 1.0
    wf["114"]["inputs"]["noise_seed"] = int(seed) + 1

    progress(0.1, desc="Preparing Inputs...")
    if mode == "Text-to-Video":
        wf["290"]["inputs"]["value"] = True
        dummy_name = "dummy_t2v.jpg"
        Image.new("RGB", (W, H), "black").save(os.path.join(INPUT_PATH, dummy_name))
        wf["167"]["inputs"]["image"] = dummy_name
    else:
        if image_filepath is None:
            raise gr.Error("Please upload an image for Image-to-Video.")
        wf["290"]["inputs"]["value"] = False
        filename = os.path.basename(image_filepath)
        shutil.copy(image_filepath, os.path.join(INPUT_PATH, filename))
        wf["167"]["inputs"]["image"] = filename

    progress(0.2, desc="Queuing Generation...")
    pid = queue_prompt(wf)["prompt_id"]
    progress(0.3, desc="Rendering Video. This can take a while...")

    while True:
        try:
            h = json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}").read())
            if str(pid) in h: 
                break
            q = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/queue").read())
            if not any(str(j[1]) == str(pid) for j in q.get("queue_running", []) + q.get("queue_pending", [])):
                raise gr.Error("Generation failed or crashed.")
        except Exception as e:
            if "Generation failed" in str(e):
                raise e
        time.sleep(3)

    progress(1.0, desc="Done!")
    return get_latest_video()

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🎬 LTX-2.3 Video Generator — Lightning AI")
    gr.Markdown(f"Using dev GGUF model with distilled LoRA v1.1: `{LORA_11_NAME}`")
    with gr.Row():
        with gr.Column(scale=1):
            mode_selector = gr.Radio(["Text-to-Video", "Image-to-Video"], value="Text-to-Video", label="Generation Mode")
            image_input = gr.Image(type="filepath", label="Upload Starting Image", visible=False)
            prompt_input = gr.Textbox(label="Prompt", placeholder="A cinematic shot...", lines=3)
            with gr.Row():
                width_slider = gr.Slider(minimum=256, maximum=1920, step=32, value=832, label="Width")
                height_slider = gr.Slider(minimum=256, maximum=1080, step=32, value=480, label="Height")
            with gr.Row():
                duration_slider = gr.Slider(minimum=1, maximum=10, step=1, value=3, label="Duration (s)")
                seed_input = gr.Number(value=43, label="Seed", precision=0)
            generate_btn = gr.Button("▶ Generate Video", variant="primary")
        with gr.Column(scale=1):
            video_output = gr.Video(label="Generated Output")

    def update_visibility(mode):
        return gr.update(visible=(mode == "Image-to-Video"))

    mode_selector.change(fn=update_visibility, inputs=mode_selector, outputs=image_input)
    generate_btn.click(fn=generate_video, inputs=[mode_selector, image_input, prompt_input, width_slider, height_slider, duration_slider, seed_input], outputs=video_output)

demo.launch(share=True, inline=True)
